## Environment Setup

```bash
# Create conda environment from the provided file
conda env create -f environment.yml

# Activate the environment
conda activate patrick_rnaseq

# Register the Jupyter kernel
python -m ipykernel install --user --name patrick_rnaseq --display-name "Patrick RNA-seq"
```

Then open this notebook and select the **Patrick RNA-seq** kernel.

# Ribo-seq / RNA-seq Integrated Analysis

**Dataset:** GEO Series - Human tumor and normal ribosome profiling + RNA-seq  
**Reference:** GRCh38.p14 (GENCODE v46)

| SRA ID | GEO ID | Sample | Type |
|--------|--------|--------|------|
| SRR1562541 | - | Unknown A | riboseq |
| SRR1562542 | - | Unknown B | riboseq |
| SRR1562543 | GSM1495248 | Tumor Ribo-seq B | riboseq |
| SRR1562544 | GSM1495249 | Normal RNA-seq A | rnaseq |
| SRR1562545 | GSM1495250 | Normal RNA-seq B | rnaseq |
| SRR1562546 | GSM1495251 | Normal RNA-seq C | rnaseq |
| SRR1562547 | GSM1495252 | Tumor RNA-seq A | rnaseq |
| SRR1562548 | GSM1495253 | Tumor RNA-seq B | rnaseq |

**Goals:**
1. Gene-level vs sequence-level read count visualization
2. Translational efficiency (TE = Ribo-seq RPF / RNA-seq mRNA)
3. 3'UTR coverage heatmaps to identify regulatory motifs and read-through events

## 1. Setup and Configuration

In [ ]:
import json
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import pysam
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE = Path("/Volumes/Felix_SSD/Patrick")
REF_DIR = BASE / "reference"
STAR_INDEX = REF_DIR / "STAR_index"
GENOME_FA = REF_DIR / "GRCh38.primary_assembly.genome.fa"
GTF = REF_DIR / "gencode.v46.annotation.gtf"
TRIM_DIR = BASE / "trimmed"
ALIGN_DIR = BASE / "aligned"

for d in [REF_DIR, TRIM_DIR, ALIGN_DIR]:
    d.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# Sample metadata (all 8 samples)
# ---------------------------------------------------------------------------
SAMPLES = {
    "SRR1562541": {"name": "Unknown Ribo-seq A",  "type": "riboseq"},
    "SRR1562542": {"name": "Unknown Ribo-seq B",  "type": "riboseq"},
    "SRR1562543": {"name": "Tumor Ribo-seq B",    "type": "riboseq"},
    "SRR1562544": {"name": "Normal RNA-seq A",    "type": "rnaseq"},
    "SRR1562545": {"name": "Normal RNA-seq B",    "type": "rnaseq"},
    "SRR1562546": {"name": "Normal RNA-seq C",    "type": "rnaseq"},
    "SRR1562547": {"name": "Tumor RNA-seq A",     "type": "rnaseq"},
    "SRR1562548": {"name": "Tumor RNA-seq B",     "type": "rnaseq"},
}


def find_fastq(sra_id):
    """Find the FASTQ file for a given SRA ID (handles variable naming)."""
    for p in BASE.glob("*.fastq"):
        if sra_id in p.name:
            return p
    return None


# ---------------------------------------------------------------------------
# Plotly publication template
# ---------------------------------------------------------------------------
PALETTE = [
    "rgb(102,197,204)", "rgb(246,207,113)", "rgb(248,156,116)",
    "rgb(220,176,242)", "rgb(135,197,95)",  "rgb(158,185,243)",
    "rgb(254,136,177)", "rgb(201,219,116)",
]
SAMPLE_COLORS = {sra: PALETTE[i] for i, sra in enumerate(SAMPLES)}
TYPE_COLORS = {"riboseq": "rgb(102,197,204)", "rnaseq": "rgb(246,207,113)"}


def pub_style(fig, width=1000, height=600):
    """Apply publication-quality styling."""
    fig.update_layout(
        plot_bgcolor="white", paper_bgcolor="white",
        width=width, height=height,
        font=dict(family="Arial", size=13, color="black"),
    )
    fig.update_xaxes(showline=True, linewidth=2, linecolor="black",
                     tickcolor="black", showgrid=False)
    fig.update_yaxes(showline=True, linewidth=2, linecolor="black",
                     tickcolor="black", showgrid=True, gridcolor="lightgray")
    return fig


print(f"Samples: {len(SAMPLES)}")
for sra, meta in SAMPLES.items():
    fq = find_fastq(sra)
    print(f"  {sra} [{meta['type']:7s}] {meta['name']:25s} {'FOUND' if fq else 'MISSING'}")

## 2. Reference Genome Download and Indexing

Download GRCh38.p14 primary assembly and GENCODE v46 annotation, then build the
STAR genome index. Each step is skipped if the output already exists.

In [ ]:
# Download genome FASTA and GTF from GENCODE
GENCODE_BASE = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_46"

downloads = {
    GENOME_FA.with_suffix(".fa.gz"): f"{GENCODE_BASE}/GRCh38.primary_assembly.genome.fa.gz",
    GTF.with_suffix(".gtf.gz"):     f"{GENCODE_BASE}/gencode.v46.annotation.gtf.gz",
}

for local_path, url in downloads.items():
    if local_path.exists() or local_path.with_suffix("").exists():
        print(f"  Exists: {local_path.name}")
        continue
    print(f"  Downloading {local_path.name}...")
    subprocess.run(["curl", "-L", "-o", str(local_path), url], check=True)

# Decompress if needed
for gz in [GENOME_FA.with_suffix(".fa.gz"), GTF.with_suffix(".gtf.gz")]:
    decompressed = gz.with_suffix("")
    if not decompressed.exists() and gz.exists():
        print(f"  Decompressing {gz.name}...")
        subprocess.run(["gunzip", "-k", str(gz)], check=True)
    else:
        print(f"  Ready: {decompressed.name}")

print("Reference files ready.")

In [ ]:
# Build STAR genome index (requires ~32 GB RAM, takes 20-40 min)
STAR_INDEX.mkdir(exist_ok=True)

if (STAR_INDEX / "SA").exists():
    print("STAR index already exists, skipping.")
else:
    print("Building STAR index (this will take 20-40 minutes)...")
    cmd = [
        "STAR", "--runMode", "genomeGenerate",
        "--genomeDir", str(STAR_INDEX),
        "--genomeFastaFiles", str(GENOME_FA),
        "--sjdbGTFfile", str(GTF),
        "--runThreadN", "8",
        "--sjdbOverhang", "100",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[-500:]}")
    else:
        print("STAR index complete.")

## 3. Adapter Trimming (fastp)

Trim adapters and filter reads. Ribo-seq samples get a length filter of 20-35 nt
(ribosome-protected fragment size). RNA-seq samples get a minimum length of 20 nt.

In [ ]:
def run_fastp(sra_id, sample_type):
    """Trim one sample with fastp. Skips if output exists."""
    out_fq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if out_fq.exists():
        print(f"  {sra_id}: trimmed file exists, skipping")
        return

    in_fq = find_fastq(sra_id)
    if in_fq is None:
        print(f"  {sra_id}: FASTQ not found, skipping")
        return

    cmd = [
        "fastp",
        "-i", str(in_fq),
        "-o", str(out_fq),
        "--html", str(TRIM_DIR / f"{sra_id}_fastp.html"),
        "--json", str(TRIM_DIR / f"{sra_id}_fastp.json"),
        "--thread", "8",
        "--length_required", "20",
    ]
    if sample_type == "riboseq":
        cmd += ["--length_limit", "35"]

    print(f"  {sra_id}: trimming ({sample_type})...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[-300:]}")
    else:
        print(f"  {sra_id}: done")


for sra_id, meta in SAMPLES.items():
    run_fastp(sra_id, meta["type"])

## 4. QC Summary

Parse fastp JSON reports to compare quality metrics across all samples.

In [ ]:
# Parse fastp JSON reports for all samples
qc_rows = []
for sra_id, meta in SAMPLES.items():
    json_path = TRIM_DIR / f"{sra_id}_fastp.json"
    if not json_path.exists():
        continue
    with open(json_path) as f:
        rpt = json.load(f)

    before = rpt["summary"]["before_filtering"]
    after = rpt["summary"]["after_filtering"]
    filt = rpt["filtering_result"]
    dup_rate = rpt.get("duplication", {}).get("rate", 0)
    adapter_trimmed = rpt.get("adapter_cutting", {}).get("adapter_trimmed_reads", 0)

    qc_rows.append({
        "Sample": meta["name"],
        "SRA": sra_id,
        "Type": meta["type"],
        "Raw reads (M)": before["total_reads"] / 1e6,
        "Passed reads (M)": after["total_reads"] / 1e6,
        "Pass rate (%)": filt["passed_filter_reads"] / before["total_reads"] * 100,
        "Adapter trimmed (M)": adapter_trimmed / 1e6,
        "Duplication (%)": dup_rate * 100,
        "GC (%)": before["gc_content"] * 100,
    })

qc_df = pd.DataFrame(qc_rows)
qc_df

In [ ]:
# QC bar chart: raw vs passed reads per sample
fig = go.Figure()
fig.add_trace(go.Bar(
    name="Raw reads",
    x=qc_df["Sample"], y=qc_df["Raw reads (M)"],
    marker_color="lightgray",
))
fig.add_trace(go.Bar(
    name="Passed filter",
    x=qc_df["Sample"], y=qc_df["Passed reads (M)"],
    marker_color=[SAMPLE_COLORS[s] for s in qc_df["SRA"]],
))
fig.update_layout(
    title="Read counts before and after trimming",
    yaxis_title="Reads (millions)",
    barmode="group",
    xaxis_tickangle=-30,
)
pub_style(fig)
fig.show()

## 5. STAR Alignment

Align trimmed reads to GRCh38.p14 using STAR. Samples run **sequentially** because
STAR loads the full genome index into shared memory (~30 GB).

- **Ribo-seq**: `--alignIntronMax 1` (RPFs should not span introns), stricter mismatch/multimap filters
- **RNA-seq**: default splice-aware parameters

In [ ]:
def run_star(sra_id, sample_type):
    """Run STAR alignment for one sample. Skips if BAM already exists."""
    out_prefix = str(ALIGN_DIR / f"{sra_id}_")
    bam_sorted = ALIGN_DIR / f"{sra_id}_Aligned.sortedByCoord.out.bam"

    if bam_sorted.exists() and bam_sorted.stat().st_size > 0:
        print(f"  {sra_id}: BAM exists ({bam_sorted.stat().st_size / 1e9:.1f} GB), skipping")
        return

    fastq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if not fastq.exists():
        print(f"  {sra_id}: trimmed FASTQ not found, skipping")
        return

    cmd = [
        "STAR",
        "--runThreadN", "8",
        "--genomeDir", str(STAR_INDEX),
        "--readFilesIn", str(fastq),
        "--readFilesCommand", "zcat",
        "--outSAMtype", "BAM", "SortedByCoordinate",
        "--outFileNamePrefix", out_prefix,
        "--quantMode", "GeneCounts",
        "--outSAMattributes", "NH", "HI", "AS", "NM", "MD",
    ]

    if sample_type == "riboseq":
        cmd += [
            "--alignIntronMax", "1",
            "--outFilterMismatchNmax", "2",
            "--outFilterMultimapNmax", "1",
        ]

    print(f"  {sra_id}: Running STAR ({sample_type})...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[-500:]}")
        return

    # Index the BAM
    subprocess.run(["samtools", "index", str(bam_sorted)])
    print(f"  {sra_id}: done ({bam_sorted.stat().st_size / 1e9:.1f} GB)")


# Run sequentially to avoid exceeding RAM
for sra_id, meta in SAMPLES.items():
    run_star(sra_id, meta["type"])

In [ ]:
# Parse STAR Log.final.out for alignment statistics
def parse_star_log(sra_id):
    log_path = ALIGN_DIR / f"{sra_id}_Log.final.out"
    if not log_path.exists():
        return None
    stats = {}
    with open(log_path) as f:
        for line in f:
            if "|" in line:
                key, val = line.split("|", 1)
                key = key.strip()
                val = val.strip().rstrip("%")
                try:
                    stats[key] = float(val)
                except ValueError:
                    stats[key] = val
    return stats


align_rows = []
for sra_id, meta in SAMPLES.items():
    s = parse_star_log(sra_id)
    if s is None:
        continue
    align_rows.append({
        "Sample": meta["name"],
        "SRA": sra_id,
        "Type": meta["type"],
        "Input reads (M)": s.get("Number of input reads", 0) / 1e6,
        "Uniquely mapped (%)": s.get("Uniquely mapped reads %", 0),
        "Multi-mapped (%)": s.get("% of reads mapped to multiple loci", 0),
        "Unmapped too short (%)": s.get("% of reads unmapped: too short", 0),
    })

align_df = pd.DataFrame(align_rows)

# Stacked bar: mapping categories
fig = go.Figure()
for col, color in [
    ("Uniquely mapped (%)", "rgb(102,197,204)"),
    ("Multi-mapped (%)",    "rgb(246,207,113)"),
    ("Unmapped too short (%)", "rgb(248,156,116)"),
]:
    fig.add_trace(go.Bar(
        name=col.replace(" (%)", ""),
        x=align_df["Sample"], y=align_df[col],
        marker_color=color,
    ))

fig.update_layout(
    barmode="stack",
    title="STAR alignment rates",
    yaxis_title="Reads (%)",
    yaxis_range=[0, 105],
    xaxis_tickangle=-30,
)
pub_style(fig)
fig.show()
align_df

## 6. Gene-Level Counting (featureCounts)

Count reads per gene using featureCounts from the Subread package. Counts are
assigned at the gene level using GENCODE v46 exon annotation.

In [ ]:
# Run featureCounts on all BAMs
counts_file = ALIGN_DIR / "gene_counts.txt"

if not counts_file.exists():
    bams = [
        str(ALIGN_DIR / f"{s}_Aligned.sortedByCoord.out.bam")
        for s in SAMPLES
        if (ALIGN_DIR / f"{s}_Aligned.sortedByCoord.out.bam").exists()
    ]
    cmd = [
        "featureCounts",
        "-a", str(GTF),
        "-o", str(counts_file),
        "-T", "8",
        "-t", "exon",
        "-g", "gene_id",
        "--primary",
    ] + bams
    print(f"Running featureCounts on {len(bams)} BAMs...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[-500:]}")
    else:
        print("featureCounts complete.")
else:
    print("Gene counts file exists, skipping.")

In [ ]:
# Parse featureCounts output into a clean DataFrame
raw = pd.read_csv(counts_file, sep="\t", comment="#")

# Rename BAM path columns to SRA IDs
col_map = {}
for col in raw.columns:
    for sra_id in SAMPLES:
        if sra_id in col:
            col_map[col] = sra_id
raw = raw.rename(columns=col_map)

# Strip version suffix from gene IDs
raw["gene_id"] = raw["Geneid"].str.split(".").str[0]

# Build counts DataFrame
available_sra = [s for s in SAMPLES if s in raw.columns]
counts = raw.set_index("gene_id")[available_sra].copy()
counts.columns = [SAMPLES[s]["name"] for s in available_sra]

# Drop genes with zero counts across all samples
counts = counts.loc[counts.sum(axis=1) > 0]

print(f"Genes with >0 reads: {len(counts):,}")
print(f"Samples: {list(counts.columns)}")
counts.head(10)

In [ ]:
# Read count distribution: histogram per sample
fig = go.Figure()
for i, col in enumerate(counts.columns):
    log_counts = np.log10(counts[col].replace(0, np.nan).dropna())
    fig.add_trace(go.Histogram(
        x=log_counts, name=col, opacity=0.6, nbinsx=80,
        marker_color=PALETTE[i % len(PALETTE)],
    ))

fig.update_layout(
    title="Gene read count distribution (all samples)",
    xaxis_title="log10(read count)",
    yaxis_title="Number of genes",
    barmode="overlay",
)
pub_style(fig)
fig.show()

In [ ]:
# Aggregate columns by type for comparison
riboseq_cols = [SAMPLES[s]["name"] for s in available_sra if SAMPLES[s]["type"] == "riboseq"]
rnaseq_cols  = [SAMPLES[s]["name"] for s in available_sra if SAMPLES[s]["type"] == "rnaseq"]

counts["Ribo-seq avg"] = counts[riboseq_cols].mean(axis=1)
counts["RNA-seq avg"]  = counts[rnaseq_cols].mean(axis=1)

# Top 30 genes by Ribo-seq, compared to RNA-seq
top_genes = counts.nlargest(30, "Ribo-seq avg")

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Ribo-seq (avg)",
    y=top_genes.index, x=top_genes["Ribo-seq avg"],
    orientation="h", marker_color=TYPE_COLORS["riboseq"],
))
fig.add_trace(go.Bar(
    name="RNA-seq (avg)",
    y=top_genes.index, x=top_genes["RNA-seq avg"],
    orientation="h", marker_color=TYPE_COLORS["rnaseq"],
))
fig.update_layout(
    title="Top 30 genes by Ribo-seq read count",
    xaxis_title="Average read count",
    barmode="group",
)
pub_style(fig, height=800)
fig.show()

## 7. Translational Efficiency

Translational efficiency (TE) measures how efficiently an mRNA is translated:

$$TE = \frac{\text{Ribo-seq RPM}}{\text{RNA-seq RPM}}$$

- **TE > 1**: translated more efficiently than average
- **TE < 1**: post-transcriptional repression

In [ ]:
# Compute RPM and translational efficiency
ribo_rpm  = counts["Ribo-seq avg"] / counts["Ribo-seq avg"].sum() * 1e6
rnaseq_rpm = counts["RNA-seq avg"] / counts["RNA-seq avg"].sum() * 1e6

# Filter: require at least 10 reads in both
mask = (counts["Ribo-seq avg"] >= 10) & (counts["RNA-seq avg"] >= 10)
te_df = pd.DataFrame({
    "ribo_rpm": ribo_rpm[mask],
    "rnaseq_rpm": rnaseq_rpm[mask],
    "ribo_counts": counts["Ribo-seq avg"][mask],
    "rnaseq_counts": counts["RNA-seq avg"][mask],
})
te_df["TE"] = te_df["ribo_rpm"] / te_df["rnaseq_rpm"]
te_df["log2_TE"] = np.log2(te_df["TE"])
te_df = te_df.replace([np.inf, -np.inf], np.nan).dropna()

print(f"Genes with TE calculated: {len(te_df):,}")
print(f"Median TE: {te_df['TE'].median():.2f}")
print(f"High TE (>2): {(te_df['TE'] > 2).sum():,} genes")
print(f"Low TE (<0.5): {(te_df['TE'] < 0.5).sum():,} genes")

In [ ]:
# Scatter: RNA-seq vs Ribo-seq RPM colored by TE
fig = px.scatter(
    te_df.reset_index(),
    x=np.log10(te_df["rnaseq_rpm"]),
    y=np.log10(te_df["ribo_rpm"]),
    color="log2_TE",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    hover_data={"gene_id": te_df.index},
    labels={
        "x": "log10(RNA-seq RPM)",
        "y": "log10(Ribo-seq RPM)",
        "color": "log2(TE)",
    },
    title="Translational Efficiency: RNA-seq vs Ribo-seq",
)

# Diagonal line (TE = 1)
rng = [np.log10(max(te_df[["rnaseq_rpm", "ribo_rpm"]].min().min(), 0.01)),
       np.log10(te_df[["rnaseq_rpm", "ribo_rpm"]].max().max())]
fig.add_trace(go.Scatter(
    x=rng, y=rng, mode="lines",
    line=dict(dash="dash", color="gray", width=1),
    name="TE = 1", showlegend=True,
))

fig.update_traces(marker=dict(size=4, opacity=0.6), selector=dict(mode="markers"))
pub_style(fig, width=800, height=700)
fig.show()

In [ ]:
# Top and bottom 30 genes by translational efficiency
te_sorted = te_df.sort_values("TE")
extreme_te = pd.concat([te_sorted.head(30), te_sorted.tail(30)])

fig = go.Figure()
fig.add_trace(go.Bar(
    y=extreme_te.index,
    x=extreme_te["log2_TE"],
    orientation="h",
    marker_color=[
        "rgb(248,156,116)" if v < 0 else "rgb(102,197,204)"
        for v in extreme_te["log2_TE"]
    ],
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Extreme translational efficiency genes (top/bottom 30)",
    xaxis_title="log2(TE)",
)
pub_style(fig, height=900)
fig.show()

## 8. 3'UTR Coverage Heatmaps

Map read coverage across 3'UTR regions to identify:
- **Read-through events**: Ribo-seq signal extending into the 3'UTR
- **Regulatory motifs**: positional patterns in coverage

Coverage is normalized to 3'UTR length (100 bins) so UTRs of different sizes
can be compared.

In [ ]:
# Extract 3'UTR coordinates from GENCODE GTF
utr_records = []
with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        if fields[2] != "three_prime_utr":
            continue
        chrom, start, end, strand = fields[0], int(fields[3]), int(fields[4]), fields[6]
        attrs = fields[8]
        gid = re.search(r'gene_id "([^"]+)"', attrs)
        gname = re.search(r'gene_name "([^"]+)"', attrs)
        if gid:
            utr_records.append({
                "chrom": chrom, "start": start, "end": end, "strand": strand,
                "gene_id": gid.group(1).split(".")[0],
                "gene_name": gname.group(1) if gname else gid.group(1),
                "length": end - start,
            })

utr_df = pd.DataFrame(utr_records)
utr_df = utr_df.sort_values("length", ascending=False).drop_duplicates("gene_id", keep="first")
utr_df = utr_df[utr_df["length"] >= 200].reset_index(drop=True)

print(f"3'UTRs extracted: {len(utr_df):,}")
print(f"Median length: {utr_df['length'].median():.0f} bp")

In [ ]:
def compute_utr_coverage(bam_path, utr_row, n_bins=100):
    """Compute binned coverage across a 3'UTR region."""
    bam = pysam.AlignmentFile(str(bam_path), "rb")
    start = utr_row["start"] - 1
    end = utr_row["end"]
    try:
        coverage = np.zeros(end - start)
        for col in bam.pileup(utr_row["chrom"], start, end,
                              truncate=True, stepper="nofilter"):
            pos = col.reference_pos - start
            if 0 <= pos < len(coverage):
                coverage[pos] = col.nsegments
    except ValueError:
        return np.zeros(n_bins)
    finally:
        bam.close()

    if utr_row["strand"] == "-":
        coverage = coverage[::-1]

    if len(coverage) == 0:
        return np.zeros(n_bins)
    edges = np.linspace(0, len(coverage), n_bins + 1, dtype=int)
    return np.array([
        coverage[edges[i]:edges[i+1]].mean() if edges[i] < edges[i+1] else 0
        for i in range(n_bins)
    ])


def build_coverage_matrix(bam_path, utr_subset, n_bins=100):
    """Build genes x bins coverage matrix."""
    mat = np.zeros((len(utr_subset), n_bins))
    for i, (_, row) in enumerate(utr_subset.iterrows()):
        mat[i] = compute_utr_coverage(bam_path, row, n_bins)
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(utr_subset)}", end="\r")
    print(f"  Done: {len(utr_subset)} UTRs")
    return mat

In [ ]:
# Select top 500 genes by Ribo-seq counts that have a known 3'UTR
utr_te = utr_df[utr_df["gene_id"].isin(te_df.index)].copy()
utr_te = utr_te.merge(
    te_df[["ribo_counts"]].rename(columns={"ribo_counts": "ribo_total"}),
    left_on="gene_id", right_index=True,
)
utr_top = utr_te.nlargest(500, "ribo_total").reset_index(drop=True)

# Pick one Ribo-seq and one RNA-seq BAM for coverage
ribo_bam = next(
    ALIGN_DIR / f"{s}_Aligned.sortedByCoord.out.bam"
    for s in SAMPLES if SAMPLES[s]["type"] == "riboseq"
    and (ALIGN_DIR / f"{s}_Aligned.sortedByCoord.out.bam").exists()
)
rnaseq_bam = next(
    ALIGN_DIR / f"{s}_Aligned.sortedByCoord.out.bam"
    for s in SAMPLES if SAMPLES[s]["type"] == "rnaseq"
    and (ALIGN_DIR / f"{s}_Aligned.sortedByCoord.out.bam").exists()
)
print(f"Ribo-seq BAM: {ribo_bam.name}")
print(f"RNA-seq BAM:  {rnaseq_bam.name}")
print(f"Computing coverage for {len(utr_top)} UTRs...")

print("\nRibo-seq:")
ribo_matrix = build_coverage_matrix(ribo_bam, utr_top)
print("RNA-seq:")
rnaseq_matrix = build_coverage_matrix(rnaseq_bam, utr_top)

In [ ]:
# Normalize and sort by read-through score
def row_normalize(mat):
    mx = mat.max(axis=1, keepdims=True)
    mx[mx == 0] = 1
    return mat / mx

ribo_norm = row_normalize(ribo_matrix)
rnaseq_norm = row_normalize(rnaseq_matrix)

# Read-through score: ratio of last-third to first-third coverage
first_third = ribo_matrix[:, :33].mean(axis=1)
last_third  = ribo_matrix[:, 67:].mean(axis=1)
first_third[first_third == 0] = 0.01
rt_score = last_third / first_third

has_cov = ribo_matrix.sum(axis=1) > 0
sort_idx = np.argsort(rt_score[has_cov])[::-1]

ribo_disp   = ribo_norm[has_cov][sort_idx]
rnaseq_disp = rnaseq_norm[has_cov][sort_idx]
gene_names  = utr_top.loc[has_cov, "gene_name"].values[sort_idx]

print(f"Genes with 3'UTR coverage: {has_cov.sum()}")

In [ ]:
# Side-by-side heatmaps: Ribo-seq vs RNA-seq 3'UTR coverage
n_show = min(200, len(ribo_disp))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ribo-seq 3'UTR coverage", "RNA-seq 3'UTR coverage"),
    shared_yaxes=True,
    horizontal_spacing=0.05,
)

x_labels = [f"{i}%" for i in range(100)]

fig.add_trace(go.Heatmap(
    z=ribo_disp[:n_show], x=x_labels, y=gene_names[:n_show],
    colorscale="YlOrRd",
    colorbar=dict(title="Norm.", x=0.46, len=0.8),
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=rnaseq_disp[:n_show], x=x_labels, y=gene_names[:n_show],
    colorscale="YlGnBu",
    colorbar=dict(title="Norm.", x=1.02, len=0.8),
), row=1, col=2)

fig.update_layout(
    title="3'UTR coverage heatmap (sorted by read-through score)",
    height=max(600, n_show * 4), width=1200,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=11, color="black"),
)
fig.update_xaxes(title_text="3'UTR position (5' -> 3')", row=1, col=1)
fig.update_xaxes(title_text="3'UTR position (5' -> 3')", row=1, col=2)
fig.update_yaxes(title_text="Gene", row=1, col=1, autorange="reversed")
fig.show()

### Read-through Candidates

Genes at the top of the heatmap show sustained Ribo-seq signal into the 3'UTR,
suggesting stop codon read-through or re-initiation. Compare against the RNA-seq
panel to distinguish genuine translational read-through from mRNA coverage.

In [ ]:
# Top read-through candidates
rt_df = pd.DataFrame({
    "gene_name": gene_names,
    "readthrough_score": rt_score[has_cov][sort_idx],
    "ribo_3utr_mean": ribo_matrix[has_cov][sort_idx].mean(axis=1),
    "rnaseq_3utr_mean": rnaseq_matrix[has_cov][sort_idx].mean(axis=1),
})
rt_df = rt_df[rt_df["ribo_3utr_mean"] > 1]

print(f"Read-through candidates (top 20):")
rt_df.head(20)